In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

# Standard library - import pathlib with explicit alias to avoid matplotlib.path conflict
import sys
import time
import re
import random
from pathlib import Path as PathLib

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import sklearn
from sklearn.cluster import KMeans, DBSCAN
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import umap
import anndata as ad
import scanpy as sc
from sknetwork.clustering import Louvain, Leiden
from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
import xgboost as xgb
import distinctipy
import networkx
from leidenalg import find_partition
import shap
import icecream as ic

# Add custom module paths
cytof_base = PathLib("/Users/ronguy/Dropbox/Work/CyTOF")
sys.path.append(str(cytof_base))
sys.path.append(str(cytof_base / "Code"))
sys.path.append(str(cytof_base / "CellClasifier"))

from CyTOFHelper import *

# Jupyter/IPython magic commands
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Plotting configuration
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

params = {
    'axes.titlesize': 30,
    'legend.fontsize': 20,
    'figure.figsize': (6, 5),
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'figure.titlesize': 30
}
plt.rcParams.update(params)
sns.set_style("white")

# Analysis configuration
Run = "Corrs"
hKWD = {'element': 'step', 'fill': False, 'stat': 'density'}
pKWD = {'dpi': 200, 'bbox_inches': 'tight'}

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
from cell_classifier import label_clusters, AsyncClusterLabeler

In [ ]:
def plot_histograms_multi_df(dataframes, columns, df_names=None, ncols=3, figsize=(20, 20), 
                             colors=None, hist_kwargs=None, **kwargs):
    """
    Plot histograms for multiple dataframes, one panel per column.
    
    Parameters:
    -----------
    dataframes : list of pd.DataFrame
        List of dataframes to plot
    columns : list of str
        List of column names to plot (one histogram per column)
    df_names : list of str, optional
        Names for each dataframe (for legend). If None, uses 'DF_0', 'DF_1', etc.
    ncols : int, default=3
        Number of columns in the subplot grid
    figsize : tuple, default=(20, 20)
        Figure size (width, height)
    colors : list or dict, optional
        Colors for each dataframe. If list, should match length of dataframes.
        If dict, should map df_names to colors. If None, uses distinctipy.
    hist_kwargs : dict, optional
        Keyword arguments to pass to sns.histplot (e.g., {'element': 'step', 'stat': 'density'})
        If None, uses default: {'element': 'step', 'fill': False, 'stat': 'density'}
    **kwargs : additional keyword arguments
        Additional arguments passed to plt.tight_layout()
    
    Returns:
    --------
    fig : matplotlib.figure.Figure
        The figure object
    axes : numpy.ndarray
        Array of axes objects
    """
    n_plots = len(columns)
    nrows = int(np.ceil(n_plots / ncols))
    
    # Create subplot grid
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize)
    
    # Flatten axes array for easier indexing
    if n_plots == 1:
        axes = [axs]
    elif nrows == 1:
        axes = axs if isinstance(axs, np.ndarray) else [axs]
    else:
        axes = axs.flatten()
    
    # Generate or use provided colors
    if colors is None:
        colors = distinctipy.get_colors(len(dataframes))
    elif isinstance(colors, dict):
        # If colors is a dict, convert to list in same order as dataframes
        if df_names is None:
            df_names = [f'DF_{i}' for i in range(len(dataframes))]
        colors = [colors.get(name, 'gray') for name in df_names]
    
    # Ensure colors is a list
    if not isinstance(colors, (list, tuple)):
        colors = list(colors)
    
    # Generate dataframe names if not provided
    if df_names is None:
        df_names = [f'DF_{i}' for i in range(len(dataframes))]
    
    # Default histogram kwargs
    if hist_kwargs is None:
        hist_kwargs = {'element': 'step', 'fill': False, 'stat': 'density'}
    
    # Plot histograms
    for i, col in enumerate(columns):
        ax = axes[i]
        
        # Plot each dataframe
        for df_idx, df in enumerate(dataframes):
            if col in df.columns:
                sns.histplot(
                    data=df,
                    x=col,
                    ax=ax,
                    color=colors[df_idx],
                    label=df_names[df_idx],
                    **hist_kwargs
                )
        
        # Set title to column name
        ax.set_title(col)
    
    # Hide unused subplots
    for i in range(n_plots, len(axes)):
        axes[i].set_visible(False)
    
    # Add legend to the side of the figure
    # Get handles and labels from the first subplot that has data
    handles, labels = None, None
    for ax in axes:
        if ax.get_legend() is not None:
            handles, labels = ax.get_legend_handles_labels()
            break
    
    # If no legend found, create one from the first subplot
    if handles is None and len(axes) > 0:
        handles, labels = axes[0].get_legend_handles_labels()
    
    # Place legend to the right side of the figure
    if handles and labels:
        fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.02, 0.5))
    
    # Use tight_layout with optional kwargs
    default_tight = {'pad': 1.0}
    default_tight.update(kwargs)
    plt.tight_layout(**default_tight)
    
    return fig, axes

# Load and initialize

In [ ]:
# Define data directory using pathlib
data_dir = PathLib("/Users/ronguy/Dropbox/IDH_CyTOF/DKFZ_project_IDH/20251113_32D_timeLaps1/output")

In [ ]:
# Use pathlib glob to get file list
FList = list(data_dir.glob("*"))

In [ ]:
# Sort files by numerical identifier
FList = sorted(FList, key=lambda s: int(re.search(r"[cC](\d+)", s.name).group(1)))

In [ ]:
DBs=[f"c{x:02}" for x in range(1,17)]

In [ ]:
DBs

In [ ]:
R={'c01':'WT',
 'c02':'WT_8h',
 'c03':'WT_24h',
 'c04':'WT_72h',
 'c05':'Mut',
 'c06':'Mut_8h',
 'c07':'Mut_24h',
 'c08':'Mut_72h',
 'c09':'WT_Ac',
 'c10':'WT_Ac_8h',
 'c11':'WT_Ac_24h',
 'c12':'WT_AC_72h',
 'c13':'Mut_Ac',
 'c14':'Mut_Ac_8h',
 'c15':'Mut_Ac_24h',
 'c16':'Mut_Ac_72h'}

In [ ]:
DBs=[R[DB] for DB in DBs]

In [ ]:
for F,DB in zip(FList,DBs):
    print(F)
    globals()[DB]=pd.read_csv(F)

In [ ]:
# Load mapping file using pathlib
mapping_file = PathLib("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx")
Rep=dict(pd.read_excel(mapping_file).iloc[:,:].values)

In [ ]:
Rep

In [ ]:
for F,DB in zip(FList,DBs):
    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
#    globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)

In [ ]:
N=list(WT.columns)
N.sort()

In [ ]:
N

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
plot_histograms_multi_df([np.arcsinh(globals()[db]/5) for db in DBs],
                         N,df_names=DBs,hist_kwargs=hKWD,ncols=4)

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
    sns.histplot(data=D,x='H4',**hKWD,color='magenta')
#    sns.histplot(data=D,x='H2A',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

## Gate on H3.3/H2A too low, but also remove outliers 99.99% from all

In [ ]:
GateColumns=['H3.3','H4','H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
#    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data

In [ ]:
for DB in DBs:
    globals()[DB]=Gate(globals()[DB],DB)

In [ ]:
plot_histograms_multi_df([globals()[db] for db in DBs],
                         N,df_names=DBs,hist_kwargs=hKWD,ncols=4);

# Normalize using new method on all intercellular markers

In [ ]:
def normalize_data(data, norm_columns, norm_markers=None):
    """
    Normalize data using a weighted combination of normalization columns.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Input data to normalize
    norm_columns : list of str
        List of column names to use for normalization (1-4 columns)
        Examples: ['H3.3'], ['H3.3', 'H4'], ['H3.3', 'H4', 'H2A'], etc.
    norm_markers : list of str, optional
        List of markers to normalize. If None, uses all columns except norm_columns.
        Default: None (uses NormMRK from global scope if available)
    
    Returns:
    --------
    pd.DataFrame
        Normalized data
    """
    if len(norm_columns) < 1 or len(norm_columns) > 4:
        raise ValueError("norm_columns must contain 1-4 column names")
    
    # Use provided norm_markers or try to use global NormMRK
    if norm_markers is None:
        try:
            norm_markers = NormMRK
        except NameError:
            # If NormMRK not available, use all columns except norm_columns
            norm_markers = [col for col in data.columns if col not in norm_columns]
    
    ddf = data.copy()
    ddf2 = data.copy()
    
    # Case 1: Single column - no optimization needed
    if len(norm_columns) == 1:
        Q = ddf[norm_markers].mean()
        M = (ddf / Q)[norm_columns[0]]
        ddf[norm_markers] = ddf[norm_markers].divide(M, axis=0).copy()
        ddf2[norm_markers] = ddf[norm_markers]
        print(f"Single column normalization using {norm_columns[0]}")
        print(f"Shape: {ddf2.shape}")
        return ddf2
    
    # Case 2-4: Multiple columns - need optimization
    Q = ddf[norm_markers].mean()
    M_list = [(ddf / Q)[col] for col in norm_columns]
    
    # Create objective function dynamically
    n_params = len(norm_columns) - 1
    
    def create_objective_function(n_cols, cols_list):
        """Create objective function for n columns."""
        def objective(p, x, data, Q, M_list):
            # Get parameter values
            param_values = [p[f'p{i}'].value for i in range(n_cols - 1)]
            # Last weight is 1 - sum of others
            last_weight = 1 - sum(param_values)
            
            # Ensure weights sum to 1 and are non-negative
            if last_weight < 0:
                return 1e10  # Penalty for invalid weights
            
            # Build weighted combination
            M_combined = param_values[0] * M_list[0]
            for i in range(1, n_cols - 1):
                M_combined += param_values[i] * M_list[i]
            M_combined += last_weight * M_list[-1]
            
            # Divide and compute sum of squared standard deviations
            d = x.divide(M_combined, axis=0)
            return sum(d.std()[col]**2 for col in cols_list)
        
        return objective
    
    # Create parameters
    params = Parameters()
    param_names = [f'p{i}' for i in range(n_params)]
    
    # Set initial values and bounds
    # For 2 columns: first param in [0.1, 1.0] (matching NormalizeNew2)
    # For 3+ columns: first param in [0.1, 0.9], others in [0, 0.9]
    # This ensures sum can't exceed 1
    initial_value = 1.0 / len(norm_columns)  # Start with equal weights
    for i, pname in enumerate(param_names):
        if i == 0:
            if len(norm_columns) == 2:
                params.add(pname, value=initial_value, min=0.0, max=1.0)
            else:
                params.add(pname, value=initial_value, min=0.0, max=0.9)
        else:
            params.add(pname, value=initial_value, min=0, max=0.9)
    
    # Create and minimize objective function
    R = create_objective_function(len(norm_columns), norm_columns)
    out = minimize(R, params, args=(ddf[norm_markers], ddf[norm_markers], Q, M_list), method='cg')
    
    # Extract optimized parameters
    param_values = [out.params[pname].value for pname in param_names]
    last_weight = 1 - sum(param_values)
    
    # Build final weighted combination
    M = param_values[0] * M_list[0]
    for i in range(1, n_params):
        M += param_values[i] * M_list[i]
    M += last_weight * M_list[-1]
    
    # Apply normalization
    ddf[norm_markers] = ddf[norm_markers].divide(M, axis=0).copy()
    ddf2[norm_markers] = ddf[norm_markers]
    
    # Print results
    print(f"Normalization using columns: {norm_columns}")
    print(f"Optimized weights: {param_values + [last_weight]}")
    print(f"Shape: {ddf2.shape}")
    
    del ddf
    return ddf2


# Backward compatibility aliases
def NormalizeNew(data):
    """Legacy function for 3-column normalization."""
    return normalize_data(data, ['H3.3', 'H4', 'H2A'])

def NormalizeNew2(data):
    """Legacy function for 2-column normalization."""
    return normalize_data(data, ['H3.3', 'H4'])

In [ ]:
NormMRK

In [ ]:
for DB in DBs:
    globals()[DB]=normalize_data(globals()[DB],norm_columns=['H3','H3.3','H4'],norm_markers=NormMRK)

In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)

In [ ]:
MRK_All=NamesAll.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')
#MRK_All.remove('H2A')

EPC=EpiCols.copy()
Core=['H3','H3.3','H4']#,'H2A']
for C in Core:
    EPC.remove(C)

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC, replace=False, random_state=RANDOM_SEED)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Line']=DB

In [ ]:
DBCLR=dict(zip(DBs,distinctipy.get_colors(len(DBs))))

In [ ]:
plot_histograms_multi_df([globals()[db] for db in DBs],
                         MRK_All,df_names=DBs,hist_kwargs=hKWD,ncols=4);

In [ ]:
MRK_All=[
 
 'H3K27ac',
 'H3K27me3',
 'H3K4me1',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me3',
 
 'H4K16ac',
 'KI67',
 
 'mCD11b',
 'mLy-6G',
 'pRb']

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()

In [ ]:
Mat=CAll.groupby('Line').mean()

In [ ]:
Mat=Mat.loc[DBs,:]

In [ ]:
plt.figure(figsize=(5,10))
sns.clustermap(np.round(Mat[MRK_All].T,2),annot=True,cmap=plt.cm.seismic,center=0,yticklabels=True,col_cluster=True)
plt.xticks(fontsize=12);
plt.yticks(fontsize=12);
#plt.savefig(f'Plots/{Run}_All.png',**pKWD)

In [ ]:
WDBs=['WT',
 'WT_8h',
 'WT_24h',
 'WT_72h',
 'Mut',
 'Mut_8h',
 'Mut_24h',
 'Mut_72h',
]

In [ ]:
NC=2000
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in WDBs:
    CAll=pd.concat([CAll,globals()[DB].sample(NC, replace=False, random_state=RANDOM_SEED)]).copy()

In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=160,random_state=42,verbose=True)

X_2d=UM.fit_transform(CAll[MRK_All])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[NamesAll],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
MRK=NamesAll.copy()
MRK=list(set(MRK).difference(set(['H3','H3.3','H4','Line'])))

In [ ]:
MRK.sort()
AD=ad.AnnData(CAll[MRK],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
sc.pp.neighbors(AD, random_state=RANDOM_SEED)

In [ ]:
sc.tl.leiden(AD, resolution=0.75, flavor="igraph", n_iterations=2, random_state=RANDOM_SEED)

In [ ]:
sc.pl.umap(AD,color=MRK+['Line','leiden'],cmap='seismic',vmin='p01',vmax='p99',hspace=0.5)

In [ ]:
lbl=AD.obs['leiden'].astype(int).values

In [ ]:
lbl

In [ ]:
# Define merges and apply
merge_dict = {2: 0, 10: 0,  # → 0
              5: 3, 6: 3, 9: 3,  # → 3
              8: 4}  # → 4

lbl_merged = lbl.copy()
for old, new in merge_dict.items():
    lbl_merged[lbl == old] = new

# Reindex to 0, 1, 2, ...
lbl = pd.Categorical(lbl_merged).codes
# lbl[(lbl==10) | (lbl==2)]=0
# lbl[(lbl==6) | (lbl==9) | (lbl==5)]=3
# lbl[(lbl==8)]=4


In [ ]:
AD.obs['lbl']=lbl
AD.obs['lbl']=AD.obs['lbl'].astype('category')

In [ ]:
sc.pl.umap(AD,color=MRK+['Line','leiden','lbl'],
           cmap='seismic',vcenter=0,vmin='p01',vmax='p99',wspace=0.5)

In [ ]:
DF=AD.to_df()
DF['lbl']=AD.obs['lbl'].astype(int)
DF['Line']=AD.obs['Line']

In [ ]:
CM=pd.crosstab(DF['Line'],DF['lbl'])

In [ ]:
sns.heatmap(CM/CM.sum(0),annot=True,cmap='magma')

In [ ]:
CM = pd.crosstab(DF['Line'], DF['lbl'])
CM_normalized = CM / CM.sum(0)

# Stacked barplot - transpose so columns become bars
ax = CM_normalized.T.plot(kind='bar', stacked=True, cmap='magma', figsize=(10, 6),
                          edgecolor='white', linewidth=0.5)

# Add annotations on each segment
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='center', fontsize=8)

plt.ylabel('Proportion')
plt.xlabel('lbl')
plt.xticks(rotation=0)
plt.legend(title='Line', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig,ax=plt.subplots(4,4,figsize=(20,20))
a=ax.flatten()
for i,L in enumerate(CAll['Line'].unique()):
    M=CAll['Line']==L
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=L)
    a[i].legend()
    
#plt.legend()

In [ ]:
# # Run asynchronously
# try:
#     result = await label_clusters(AD, cluster_key='lbl', log_file='demo_log.txt')
#     print(result)
# except Exception as e:
#     print(f"Agent run failed (expected if no keys): {e}")

sc.pl.umap(AD,color=MRK+['Line','lbl','AI_Label'],
           cmap='seismic',vcenter=0,vmin='p01',vmax='p99',wspace=0.5
           )

In [ ]:
from CellIden_AI import *


In [ ]:
AD.write_h5ad('Data/AD_IDH_CyTOF.h5ad')

In [ ]:
from Segmenter import segment_anndata

# Using ViT-H (recommended)
app = segment_anndata(
    AD, 
    sam_ckpt='sam_vit_b_01ec64.pth',
    feature='mCD11b',
    model_type='vit_b'  # This is the default
)



In [ ]:
from segment_anything import sam_model_registry
sam = sam_model_registry["vit_b"]#(checkpoint="<path/to/chec

In [ ]:
sam()